In [1]:
import scanpy as sc
import harmonypy as hm
import tqdm as notebook_tqdm
import phate
import pandas as pd
from matplotlib.colors import to_rgba
from pygam import LinearGAM, s
import meld
import scipy.stats as stats
import numpy as np
import sklearn as sk
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
from itertools import combinations
import gseapy as gp
from scipy.stats import ttest_ind
from statannotations.Annotator import Annotator
import scprep

In [2]:
adata= sc.read_h5ad("/Users/Jan/data/final_final_data.h5ad")

In [ ]:
prediction_data= "C:/Users/Jan/cytotrace2/cytotrace2_python/cytotrace2_results/cytotrace2_results.txt"
prediction_data = pd.read_csv(
    prediction_data,
    sep="\t",
    header= 0
)
prediction_data= prediction_data.set_index("Unnamed: 0")
prediction_data

In [4]:
for column in prediction_data.columns:
    adata.obs[column]= prediction_data[column]

In [ ]:
# Figure 5A
colors = ["#5E4FA2", "#66C2A5", "#E6F598", "#FEE08B", "#F46D43", "#9E0142"]
rel_labels = ["0.0 (More diff.)", "0.2", "0.4", "0.6", "0.8", "1.0 (Less diff.)"]
sc.set_figure_params(scanpy=True, fontsize=14)
plt.rc('legend',fontsize=14)
rel_cmap = matplotlib.colors.LinearSegmentedColormap.from_list("potency_cols",list(zip(np.linspace(0,1,6),colors)))
fig = sc.pl.embedding(adata, basis= "phate", color='CytoTRACE2_Relative',cmap=rel_cmap,return_fig=True,vmin=0,vmax=1,colorbar_loc=None)
ax = plt.gca()
clb = plt.colorbar(ax.get_children()[0],ax=ax,fraction=0.035);
clb.ax.set_title('Relative order',pad=15,loc='left')
clb.ax.set_yticks([0,0.2,0.4,0.6,0.8,1])
clb.ax.set_yticklabels(rel_labels,va='center')
ax.spines[['right', 'top']].set_visible(False)
ax.set_aspect(1 / ax.get_data_ratio())

In [ ]:
# Figure 5B
cell_cycle_genes = [x.strip() for x in open("C:/Users/Jan/Downloads/Tirosh_cell_cycle_genes_mouse.txt")]
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]
not_genes= [x for x in cell_cycle_genes if x not in adata.var_names]
cell_cycle_genes = [x for x in cell_cycle_genes if x in adata.var_names]
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
adata_cc_genes = adata[:, cell_cycle_genes]
sc.external.pl.phate(adata_cc_genes, color= "phase")

In [7]:
def format_string(s,max_line_length=25):
    all_chunks = s.split(' ')
    total_num_chunks = len(all_chunks)
    new_string = ''
    current_line_length = 0
    i = 0
    while i < total_num_chunks:  
        chunk = all_chunks[i]
        if len(chunk)>max_line_length:
            line = chunk[:max_line_length]+'...\n'
            new_string += line
            i += 1
        else:
            current_line_length = len(chunk)
            line = chunk
            while (current_line_length < max_line_length) and ((i+1) < total_num_chunks):
                candidate_line_length = len(line+' '+all_chunks[i+1])
                if candidate_line_length < max_line_length:
                    line += ' '+all_chunks[i+1]
                    current_line_length = len(line)
                    i += 1
                else:
                    current_line_length = max_line_length
            new_string += line+'\n'
            i += 1
            
    if new_string.endswith('\n'):
        new_string = new_string[:-1]
    
    return new_string

In [ ]:
# Figure 5C
violinfont = {'fontname':'Helvetica','fontsize':14}
xlabel_font = {'fontname':'Helvetica','fontsize':10}
pheno_median_score = adata.obs.groupby("Sample name",observed=True)['CytoTRACE2_Score'].median().sort_values()
adata.obs['CytoTRACE2_Score_Phenotype_Median'] = list(pheno_median_score.loc[adata.obs['Sample name'].values].values)
cytotrace2_pal = [plt.get_cmap('Spectral', 51)(round(max(0, min(5.5 - 6 * cytotrace2, 5)) * 10)) for cytotrace2 in pheno_median_score.values]
pheno_order= ['1-WT-No-Tx', '1-WT-1h', '1-WT-4h', '1-KA-No-Tx', '1-KA-1h', '1-KA-4h']
plt.figure(figsize=(6+len(pheno_order)/5, 6))
pheno_order_format = [format_string(s) for s in pheno_order]
ax = sns.boxplot(data=adata.obs, x='Sample name', y='CytoTRACE2_Score',hue='CytoTRACE2_Score_Phenotype_Median',showfliers=False,
                 order=pheno_order,linewidth=1,palette=cytotrace2_pal,legend=False)
ax.grid(False)
sns.stripplot(data=adata.obs, x='Sample name', y='CytoTRACE2_Score', alpha=0.25, dodge=True,
              jitter=True, size=1, order=pheno_order,legend=False, color='k')
plt.xticks(pheno_order,pheno_order_format,rotation=90,**xlabel_font)
plt.yticks(**violinfont)
plt.ylabel('Potency score',**violinfont)
plt.xlabel('Condition')
plt.ylim(0,1.0)
plt.xlim(-0.75,len(pheno_order)-0.25)
for i in range(6):
    plt.hlines((i + 1)/6, -0.75, len(pheno_order)-0.25, linestyles='-', colors='grey', linewidth=0.15)
plt.title("Developmental potential by condition",**violinfont,pad=10)
sns.despine()
plt.tight_layout()

In [ ]:
# Figure 5D
pheno_median_score = adata.obs.groupby("predicted_labels",observed=True)['CytoTRACE2_Score'].median().sort_values()
adata.obs['CytoTRACE2_Score_Phenotype_Median'] = list(pheno_median_score.loc[adata.obs['predicted_labels'].values].values)
cytotrace2_pal = [plt.get_cmap('Spectral', 51)(round(max(0, min(5.5 - 6 * cytotrace2, 5)) * 10)) for cytotrace2 in pheno_median_score.values]
violinfont = {'fontname':'Helvetica','fontsize':14}
xlabel_font = {'fontname':'Helvetica','fontsize':10}
pheno_order = ['TA 2', 'TA 1', 'proCSC', 'CSC', 'revCSC', 'Early Enterocyte', 'Late Enterocyte', 'ER Stress', 'Goblet or DCS']
plt.figure(figsize=(6+len(pheno_order)/5, 6))
pheno_order_format = [format_string(s) for s in pheno_order]
ax = sns.boxplot(data=adata.obs, x='predicted_labels', y='CytoTRACE2_Score',hue='CytoTRACE2_Score_Phenotype_Median',showfliers=False,
                 order=pheno_order,linewidth=1,palette=cytotrace2_pal,legend=False)
ax.grid(False)
sns.stripplot(data=adata.obs, x='predicted_labels', y='CytoTRACE2_Score', alpha=0.25, dodge=True,
              jitter=True, size=1, order=pheno_order,legend=False, color='k')
plt.xticks(pheno_order,pheno_order_format,rotation=90,**xlabel_font)
plt.yticks(**violinfont)
plt.ylabel('Potency score',**violinfont)
plt.xlabel('Cell type')
plt.ylim(0,1.0)
plt.xlim(-0.75,len(pheno_order)-0.25)
for i in range(6):
    plt.hlines((i + 1)/6, -0.75, len(pheno_order)-0.25, linestyles='-', colors='grey', linewidth=0.15)
plt.title("Developmental potential by cell type",**violinfont,pad=10)
sns.despine()
plt.tight_layout()

In [ ]:
# Figure 5F
pheno_order= ['1-WT-No-Tx', '1-WT-1h', '1-WT-4h', '1-KA-No-Tx', '1-KA-1h', '1-KA-4h']
adata_g1= adata[adata.obs["phase"]== "G1"]
pheno_median_score = adata_g1.obs.groupby("Sample name",observed=True)['CytoTRACE2_Score'].median().sort_values()
adata_g1.obs['CytoTRACE2_Score_Phenotype_Median'] = list(pheno_median_score.loc[adata_g1.obs['Sample name'].values].values)
cytotrace2_pal = [plt.get_cmap('Spectral', 51)(round(max(0, min(5.5 - 6 * cytotrace2, 5)) * 10)) for cytotrace2 in pheno_median_score.values]
plt.figure(figsize=(6+len(pheno_order)/5, 6))
pheno_order_format = [format_string(s) for s in pheno_order]
ax = sns.boxplot(data=adata_g1.obs, x='Sample name', y='CytoTRACE2_Score',hue='CytoTRACE2_Score_Phenotype_Median',showfliers=False,
                 order=pheno_order,linewidth=1,palette=cytotrace2_pal,legend=False)
ax.grid(False)
sns.stripplot(data=adata_g1.obs, x='Sample name', y='CytoTRACE2_Score', alpha=0.25, dodge=True,
              jitter=True, size=1, order=pheno_order,legend=False, color='k')
plt.xticks(pheno_order,pheno_order_format,rotation=90,**xlabel_font)
plt.yticks(**violinfont)
plt.ylabel('Potency score',**violinfont)
plt.xlabel('Condition')
plt.ylim(0,1.0)
plt.xlim(-0.75,len(pheno_order)-0.25)
for i in range(6):
    plt.hlines((i + 1)/6, -0.75, len(pheno_order)-0.25, linestyles='-', colors='grey', linewidth=0.15)
ax_right = ax.twinx()
ax_right.grid(False)
ax_right.set_ylim([0,1])
ax_right.set_yticks(np.linspace(1/12,11/12,6),['Differentiated','Unipotent','Oligopotent','Multipotent','Pluripotent','Totipotent'],**violinfont)
ax_right.tick_params(axis='y', left=False, right=False, labelright=True, pad=0)
plt.title("Developmental potential by condition in G1 cells",**violinfont,pad=10)
sns.despine()
plt.tight_layout()

In [ ]:
# Figure 5G
adata_s= adata[adata.obs["phase"]== "S"]
pheno_median_score = adata_s.obs.groupby("Sample name",observed=True)['CytoTRACE2_Score'].median().sort_values()
adata_s.obs['CytoTRACE2_Score_Phenotype_Median'] = list(pheno_median_score.loc[adata_s.obs['Sample name'].values].values)
cytotrace2_pal = [plt.get_cmap('Spectral', 51)(round(max(0, min(5.5 - 6 * cytotrace2, 5)) * 10)) for cytotrace2 in pheno_median_score.values]
plt.figure(figsize=(6+len(pheno_order)/5, 6))
pheno_order_format = [format_string(s) for s in pheno_order]
ax = sns.boxplot(data=adata_s.obs, x='Sample name', y='CytoTRACE2_Score',hue='CytoTRACE2_Score_Phenotype_Median',showfliers=False,
                 order=pheno_order,linewidth=1,palette=cytotrace2_pal,legend=False)
ax.grid(False)
sns.stripplot(data=adata_s.obs, x='Sample name', y='CytoTRACE2_Score', alpha=0.25, dodge=True,
              jitter=True, size=1, order=pheno_order,legend=False, color='k')
plt.xticks(pheno_order,pheno_order_format,rotation=90,**xlabel_font)
plt.yticks(**violinfont)
plt.ylabel('Potency score',**violinfont)
plt.xlabel('Condition')
plt.ylim(0,1.0)
plt.xlim(-0.75,len(pheno_order)-0.25)
for i in range(6):
    plt.hlines((i + 1)/6, -0.75, len(pheno_order)-0.25, linestyles='-', colors='grey', linewidth=0.15)
ax_right = ax.twinx()
ax_right.grid(False)
ax_right.set_ylim([0,1])

ax_right.set_yticks(np.linspace(1/12,11/12,6),['Differentiated','Unipotent','Oligopotent','Multipotent','Pluripotent','Totipotent'],**violinfont)
ax_right.tick_params(axis='y', left=False, right=False, labelright=True, pad=0)
plt.title("Developmental potential by condition in S cells",**violinfont,pad=10)
sns.despine()
plt.tight_layout()

In [ ]:
# Figure 5H
adata_g2m= adata[adata.obs["phase"]== "G2M"]
pheno_median_score = adata_g2m.obs.groupby("Sample name",observed=True)['CytoTRACE2_Score'].median().sort_values()
adata_g2m.obs['CytoTRACE2_Score_Phenotype_Median'] = list(pheno_median_score.loc[adata_g2m.obs['Sample name'].values].values)
cytotrace2_pal = [plt.get_cmap('Spectral', 51)(round(max(0, min(5.5 - 6 * cytotrace2, 5)) * 10)) for cytotrace2 in pheno_median_score.values]
plt.figure(figsize=(6+len(pheno_order)/5, 6))
pheno_order_format = [format_string(s) for s in pheno_order]
ax = sns.boxplot(data=adata_g2m.obs, x='Sample name', y='CytoTRACE2_Score',hue='CytoTRACE2_Score_Phenotype_Median',showfliers=False,
                 order=pheno_order,linewidth=1,palette=cytotrace2_pal,legend=False)
ax.grid(False)
sns.stripplot(data=adata_g2m.obs, x='Sample name', y='CytoTRACE2_Score', alpha=0.25, dodge=True,
              jitter=True, size=1, order=pheno_order,legend=False, color='k')
plt.xticks(pheno_order,pheno_order_format,rotation=90,**xlabel_font)
plt.yticks(**violinfont)
plt.ylabel('Potency score',**violinfont)
plt.xlabel('Condition')
plt.ylim(0,1.0)
plt.xlim(-0.75,len(pheno_order)-0.25)
for i in range(6):
    plt.hlines((i + 1)/6, -0.75, len(pheno_order)-0.25, linestyles='-', colors='grey', linewidth=0.15)
ax_right = ax.twinx()
ax_right.grid(False)
ax_right.set_ylim([0,1])
ax_right.set_yticks(np.linspace(1/12,11/12,6),['Differentiated','Unipotent','Oligopotent','Multipotent','Pluripotent','Totipotent'],**violinfont)
ax_right.tick_params(axis='y', left=False, right=False, labelright=True, pad=0)
plt.title("Developmental potential by condition in G2M cells",**violinfont,pad=10)
sns.despine()
plt.tight_layout()

In [ ]:
# Figure 5I
proportions= adata.obs.groupby(adata.obs["Sample name"], as_index= False)["phase"].value_counts(normalize= True)
proportions= pd.DataFrame(proportions)
pivot = proportions.pivot(
    index="Sample name",
    columns="phase",
    values="proportion"
).fillna(0)
pivot= pivot.reindex(["1-WT-No-Tx", "1-WT-1h", "1-WT-4h", "1-KA-No-Tx", "1-KA-1h", "1-KA-4h"])
plt.figure(figsize=(14, 8))
bottom = None
for phase in pivot.columns:
    plt.bar(
        pivot.index,
        pivot[phase],
        bottom=bottom,
        label=phase
    )
    if bottom is None:
        bottom = pivot[phase].copy()
    else:
        bottom += pivot[phase]
plt.xticks(rotation=90)
plt.xlabel("Condition")
plt.ylabel("Proportion")
plt.title("Proportion of cells belonging to each cell cycle phase by condition")
plt.ylim(0, 1)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title="Phase")
plt.tight_layout()
plt.show()

In [ ]:
# switch to decoupler environment
import decoupler as dc
import scanpy as sc
import igraph as ig
import matplotlib.pyplot as plt
import seaborn as sns
sc.set_figure_params(figsize=(3, 3), frameon=False)

In [ ]:
# Figure 5J
sc.tl.dendrogram(
    adata,
    groupby="Sample name",
    use_rep="X",
)
progeny = dc.op.progeny(organism="mouse")
dc.mt.ulm(data=adata, net=progeny)
progeny_score = dc.pp.get_obsm(adata=adata, key="score_ulm")
sc.pl.matrixplot(
    adata=progeny_score,
    var_names=progeny_score.var_names,
    groupby="Sample name",
    dendrogram=True,
    standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="RdBu_r",
)